# 通过时间反向传播
因为序列比较长，可能存在长程依赖， 比如1000个字符的序列， 所以需要1000个矩阵的乘法才可能得到梯度，中间容易出现梯度消失或者梯度爆炸

每个时间步的隐状态的方程式如下所示：
$$h_t = f(x_t, h_{t-1}, w_h), \quad o_t = g(h_t, w_o)$$

这里有一个链循环计算彼此的依赖：$\{ \dots, (x_{t-1}, h_{t-1}, o_{t-1}), (x_{t}, h_{t}, o_{t}), \dots\}$

在前向传播的过程中， 一次一个时间步的遍历三元组$(x_t, h_t, o_t)$

最后通过一个目标函数计算T个时间步内输出和对应标签之间的差值，作为损失函数：
$$L(x_1, ..., x_T, y_1, ..., y_T, w_h, w_o) = \frac{1}{T} \sum_{t=1}^T l(y_t, o_t)$$

## 反向传播的链式法则展开

计算目标是找到损失 $L$ 对隐藏层权重 $w_h$ 的梯度：
$$\frac{\partial L}{\partial w_h} = \frac{1}{T} \sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial w_h} = \frac{1}{T} \sum_{t=1}^T \frac{\partial l(y_t, o_t)}{\partial o_t} \frac{\partial g(h_t, w_o)}{\partial h_t} \frac{\partial h_t}{\partial w_h}$$

这里的$o, g, h$对应的是正向传播过程中隐藏时间步和输出的状态方程

最棘手的是最后一项 $\frac{\partial h_t}{\partial w_h}$ ，因为 $h_t$ 的计算中不但直接用到了 $w_h$ ，还用到了上一时刻的 $h_{t-1}$ ，而 $h_{t-1}$ 里面又包含了 $w_h$。

（这里的$w_h$是隐藏层的参数的数量）

$$\frac{\partial h_t}{\partial w_h} = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h} + \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_h}$$

上面我们得到了一个递归的梯度表达式，下面我们需要进行一些化简：
令 $a_t = \frac{\partial h_t}{\partial w_h}$ ， $b_t = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h}$ ， $c_t = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial h_{t-1}}$

上式可以先写成：

我们先得到原始的递归公式：$$a_t = b_t + c_t a_{t-1}$$

依次代入时间步有：$$a_1 = b_1$$

$$a_2 = b_2 + c_2 a_1 = b_2 + c_2 b_1$$

$$a_3 = b_3 + c_3 a_2 = b_3 + c_3 (b_2 + c_2 b_1) = b_3 + c_3 b_2 + c_3 c_2 b_1$$

所以我们可以得到这个数列的求和通式：
$$a_t = b_t + \sum_{i=1}^{t-1} \left( \prod_{j=i+1}^t c_j \right) b_i$$

代入具体数值之后可以得到求和公式:
$$\frac{\partial h_t}{\partial w_h} = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h} + \sum_{i=1}^{t-1} \left( \prod_{j=i+1}^t \frac{\partial f(x_j, h_{j-1}, w_h)}{\partial h_{j-1}} \right) \frac{\partial f(x_i, h_{i-1}, w_h)}{\partial w_h}$$

## 截断

主要是对求导过程中的链式求导去做截断：
$$\frac{\partial h_t}{\partial w_h} = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h} + \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial h_{t-1}} \frac{\partial h_{t-1}}{\partial w_h}$$

### 截断时间步
将求和终止在$\frac {\partial{h_{t-\tau}}} {\partial{w_h}}$

### 随机截断
使用一个随机变量去替换$\frac{\partial{h_t}}{\partial{w_h}}$，具体操作如下：
我们用一个新的变量 $z_t$ 来代替原来复杂的隐状态梯度 $\frac{\partial h_t}{\partial w_h}$，其递推公式定义为：$$z_t = \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial w_h} + \xi_t \frac{\partial f(x_t, h_{t-1}, w_h)}{\partial h_{t-1}} z_{t-1}$$

$\xi_t$是一个序列，数学期望是1，通常，我们会给它设定一个概率 $c$（$0 < c \le 1$）：有 $1-c$ 的概率，令 $\xi_t = 0$。有 $c$ 的概率，令 $\xi_t = c^{-1}$。

当$\xi_t = 0$时，递归计算终止在这个t时间步，实现截断

## 通过时间反向传播的全参数推导

为了简单起见，这里设置一个没有偏置参数的循环神经网络，隐藏层中的激活函数使用恒等映射$\phi(x) = x$

明确具体的权重矩阵： $W_{hx}$ (输入到隐藏层的权重)、 $W_{hh}$ (隐藏层到隐藏层的权重)、 $W_{qh}$ (隐藏层到输出层的权重)：
$$h_t = W_{hx} x_t + W_{hh} h_{t-1}, \quad o_t = W_{qh} h_t$$

目标函数的总体损失是：
$$L = \frac{1}{T} \sum_{t=1}^T l(o_t, y_t)$$

### 从输出层开始求导（求$W_{qh}$的梯度）

对于任意的时间步t，目标函数关于模型输出的微分计算如下所示：
$$\frac{\partial L}{\partial o_t} = \frac{\partial l(o_t, y_t)}{T \cdot \partial o_t}$$

根据这个，就可以计算目标函数中关于输出层中参数$W_{qh}$的梯度：
$$\frac{\partial L}{\partial W_{qh}} = \sum_{t=1}^T \text{prod} \left( \frac{\partial L}{\partial o_t}, \frac{\partial o_t}{\partial W_{qh}} \right) = \sum_{t=1}^T \frac{\partial L}{\partial o_t} h_t^\top$$

输出层的权重 $W_{qh}$ 梯度相对好算。由于 $o_t = W_{qh} h_t$ ，只需要把各个时间步上输出误差对 $o_t$ 的梯度与隐状态 $h_t$ 求外积（转置相乘），然后把所有时间步的结果累加即可。

### 计算对隐状态$h_t$的梯度

最后一步（ $T$ 时刻）的隐状态只受当前输出的影响： 
$$\frac{\partial L}{\partial h_T} = \text{prod} \left( \frac{\partial L}{\partial o_T}, \frac{\partial o_T}{\partial h_T} \right) = W_{qh}^\top \frac{\partial L}{\partial o_T}$$

但对于任意中间时刻 $t < T$ ，隐状态 $h_t$ 不仅影响当前的输出 $o_t$ ，还会顺着时间线影响下一个隐状态 $h_{t+1}$（$h_{t+1}是由h_t经过计算得到的$）：

$$\frac{\partial L}{\partial h_t} = \text{prod} \left( \frac{\partial L}{\partial h_{t+1}}, \frac{\partial h_{t+1}}{\partial h_t} \right) + \text{prod} \left( \frac{\partial L}{\partial o_t}, \frac{\partial o_t}{\partial h_t} \right) = W_{hh}^\top \frac{\partial L}{\partial h_{t+1}} + W_{qh}^\top \frac{\partial L}{\partial o_t}$$

对这个递归式展开：
$$\frac{\partial L}{\partial h_t} = \sum_{i=t}^T (W_{hh}^\top)^{T-i} W_{qh}^\top \frac{\partial L}{\partial o_{T+t-i}}$$

这里面讲解一下为什么要引入$h_t$:

全微分的原理：如果 $L = f(u, v)$，且 $u$ 和 $v$ 都由 $x$ 决定，那么 $\frac{\partial L}{\partial x} = \frac{\partial L}{\partial u}\frac{\partial u}{\partial x} + \frac{\partial L}{\partial v}\frac{\partial v}{\partial x}$。

拆解 $h_t$ 的前向传播路径：
*  $h_t$ 参与计算当前时刻的预测输出 $o_t$ ，进而直接影响当前时刻的局部损失；
*  $h_t$ 不仅影响当前，还会顺着时间线影响下一个时刻的隐状态 $h_{t+1}$ 。$h_t$通过$h_{t+1}$间接地影响了未来所有的输出和最终的总损失 $L$

### 最终导出隐藏层权重的梯度
$$\frac{\partial L}{\partial W_{hx}} = \sum_{t=1}^T \text{prod} \left( \frac{\partial L}{\partial h_t}, \frac{\partial h_t}{\partial W_{hx}} \right) = \sum_{t=1}^T \frac{\partial L}{\partial h_t} x_t^\top$$

$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=1}^T \text{prod} \left( \frac{\partial L}{\partial h_t}, \frac{\partial h_t}{\partial W_{hh}} \right) = \sum_{t=1}^T \frac{\partial L}{\partial h_t} h_{t-1}^\top$$

这里再解释一下一些偏导的维度是怎么得到的

这里以$\frac{\partial L}{\partial o_t}$为例

在深度学习的语境下，损失函数 $L$ 始终是一个标量（Scalar），而输出 $o_t$ 是一个列向量（Column Vector），模型有 $q$ 个输出类别（比如词表大小为 $q$），那么 $o_t \in \mathbb{R}^{q \times 1}$。

一个标量对一个矩阵/向量求导，其梯度的维度必须与该矩阵/向量本身的维度完全一致。

#### 链式求导规则是怎么做到维度匹配的？
prod运算符，以$$\frac{\partial L}{\partial W_{qh}} = \text{prod} \left( \frac{\partial L}{\partial o_t}, \frac{\partial o_t}{\partial W_{qh}} \right)$$为例

这个 prod 并不是简单的矩阵乘法，而是一个“根据链式法则和维度匹配原则，自动进行必要转置和乘法”的宏操作。让我们以 $o_t = W_{qh} h_t$ 为例，解释一下维度是怎么凑齐的：
* 我们要求 $\frac{\partial L}{\partial W_{qh}}$。因为 $W_{qh} \in \mathbb{R}^{q \times h}$，所以最终的梯度矩阵也必须是 $q \times h$ 的。
* 已知量：上游传下来的梯度：$\frac{\partial L}{\partial o_t} \in \mathbb{R}^{q \times 1}$；前向传播的缓存值：$h_t \in \mathbb{R}^{h \times 1}$
* 我们要用一个 $q \times 1$ 的向量和一个 $h \times 1$ 的向量，变出一个 $q \times h$ 的矩阵。唯一的线性代数解法就是向量外积：$$\frac{\partial L}{\partial W_{qh}} = \frac{\partial L}{\partial o_t} h_t^\top$$